# KIRAVO — Free Kaggle GPU Worker\nEnable T4 x2 + Internet, then run the next cell.\n

In [ ]:
# KIRAVO — clean Kaggle worker launcher
# This cell deliberately cleans up stale Flask/Cloudflare processes first.

!pkill -f cloudflared || true
!fuser -k 7860/tcp || true
!rm -f /kaggle/working/cloudflared /kaggle/working/kiravo-tunnel.log
!wget -q https://github.com/Iamkiranofficial/Kiravoo/raw/main/kaggle/kiravo_kaggle_worker.py -O /kaggle/working/kiravo_kaggle_worker.py
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

import sys, subprocess, threading, time, re, urllib.request
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
import kiravo_kaggle_worker as worker

server_thread = threading.Thread(
    target=worker.APP.run,
    kwargs={"host": "0.0.0.0", "port": 7860, "threaded": True, "use_reloader": False},
    daemon=True,
)
server_thread.start()

healthy = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=2) as r:
            if r.status == 200:
                print("KIRAVO local worker: ONLINE")
                healthy = True
                break
    except Exception:
        time.sleep(1)

if not healthy:
    raise RuntimeError("KIRAVO worker did not bind to port 7860. Do not update Vercel yet.")

log_path = "/kaggle/working/kiravo-tunnel.log"
log_file = open(log_path, "w")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    log_text = Path(log_path).read_text(errors="ignore") if Path(log_path).exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        print("KIRAVO_WORKER_URL =", public_url)
        print("KIRAVO worker is ONLINE and tunnel is READY.")
        break
    time.sleep(2)
else:
    print("Tunnel URL not found. Cloudflare log:")
    print(Path(log_path).read_text(errors="ignore")[-4000:])

